# Vector stores and semantic search




En esta actividad se implementa un sistema básico de almacenamiento vectorial utilizando embeddings semánticos generados con SentenceTransformers.

El objetivo es comprender el funcionamiento interno de un vector store sin utilizar librerías especializadas como FAISS o LangChain.

La búsqueda semántica se realiza mediante similitud coseno entre embeddings de documentos y embeddings de consultas.

In [ ]:
import sys
!{sys.executable} -m pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer

## Part I: Basic vector store implementation

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
from sentence_transformers import SentenceTransformer

In [ ]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.model = embedding_model
        self.documents = []
        self.embeddings = None

    def add_documents(self, documents: list[Document]):
        self.documents.extend(documents)
        # Convertimos los textos a tensores
        texts = [doc.text for doc in documents]
        new_embeddings = self.model.encode(texts, convert_to_tensor=True)

        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = torch.cat((self.embeddings, new_embeddings), dim=0)

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        if not self.documents:
            return []

        # Comparamos la pregunta con lo que ya guardamos
        query_embedding = self.model.encode(query, convert_to_tensor=True)
        cos_scores = F.cosine_similarity(query_embedding.unsqueeze(0), self.embeddings)

        # Sacamos los mejores resultados
        top_results = torch.topk(cos_scores, k=min(top_k, len(self.documents)))

        results = []
        for score, idx in zip(top_results.values, top_results.indices):
            results.append(SearchResult(score.item(), self.documents[idx.item()]))

        return results

#Carga de Dataset Animal Fun Facts

In [ ]:
import pandas as pd
from google.colab import drive
from sentence_transformers import SentenceTransformer

#  Google Drive
df = pd.read_csv('/content/animal-fun-facts-dataset.csv')

# lista de documentos
documentos = []
for _, row in df.iterrows():
    # str() y .get() para evitar errores si algún dato viene vacío en el excel
    metadata = {
        'animal_name': str(row.get('animal_name', '')),
        'source': str(row.get('source', '')),
        'media_link': str(row.get('media_link', '')),
        'wikipedia_link': str(row.get('wikipedia_link', ''))
    }
    documentos.append(Document(text=str(row.get('text', '')), metadata=metadata))

# Levantar el modelo y guardar todo en VectorStore
model = SentenceTransformer('all-MiniLM-L6-v2')
vector_store = VectorStore(model)
vector_store.add_documents(documentos)

print(f"\n¡Listo! Agregados {len(documentos)} documentos al VectorStore.\n")

In [ ]:
# 5 consultas
consultas = [
    "What do polar bears eat?",
    "Tell me about very fast animals",
    "Which animals live in the ocean?",
    "Facts about birds flying",
    "How long do turtles live?"
]

# mostrar los resultados
for q in consultas:
    print(f"--- Consulta: {q} ---")
    resultados = vector_store.search(q, top_k=2)
    for res in resultados:
        animal = res.document.metadata['animal_name']
        texto = res.document.text[:70] # Cortamos el texto un poco para que no se llene la pantalla
        print(f"Similitud: {res.score:.4f} | Animal: {animal} | Texto: {texto}...")
    print("\n")

## Part II: Filtering by metadata

FilteredVectorStore extiende la funcionalidad de VectorStore agregando filtrado basado en metadatos.

Esto permite restringir las búsquedas semánticas únicamente a documentos que cumplan ciertas condiciones, por ejemplo:

- categoría
- autor
- fecha
- tipo de documento

Primero se filtran los documentos por metadata y posteriormente se calcula la similitud semántica.

In [ ]:
class FilteredVectorStore(VectorStore):
    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        if metadata_filter is None:
            return super().search(query, top_k)

        # Filtrar documentos que coincidan con el metadata_filter
        indices = [i for i, doc in enumerate(self.documents)
                   if all(doc.metadata.get(k) == v for k, v in metadata_filter.items())]

        if not indices:
            return []

        # Obtener embeddings filtrados
        filtered_embeddings = self.embeddings[indices]

        # Calcular similitud coseno
        query_embedding = self.model.encode(query, convert_to_tensor=True)
        cos_scores = F.cosine_similarity(query_embedding.unsqueeze(0), filtered_embeddings)

        # Obtener top resultados
        top_results = torch.topk(cos_scores, k=min(top_k, len(indices)))

        results = []
        for score, idx in zip(top_results.values, top_results.indices):
            original_idx = indices[idx.item()]
            results.append(SearchResult(score.item(), self.documents[original_idx]))

        return results

La similitud coseno mide qué tan similares son dos vectores en un espacio multidimensional.

Mientras más cercano sea el valor a 1, más similares son los documentos.

Esta métrica es ampliamente utilizada en sistemas de búsqueda semántica y recuperación de información.

#Carga de nuevo dataset

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
from sentence_transformers import SentenceTransformer

from datasets import load_dataset

dataset = load_dataset("ag_news")

label_map = {
    0: 'World',
    1: 'Sports',
    2: 'Business',
    3: 'Sci/Tech'
}

documents = []

for example in dataset['train'].select(range(1000)):
    documents.append(
        Document(
            text=example['text'],
            metadata={
                'category': label_map[example['label']]
            }
        )
    )

print(f'Total documents loaded: {len(documents)}')

# Crear instancia de FilteredVectorStore
filtered_vector_store = FilteredVectorStore(model)
filtered_vector_store.add_documents(documents)

print(f"Agregados {len(documents)} documentos al FilteredVectorStore.")

#5 consultas y resultados obtenidos

In [ ]:
# Consultas de ejemplo para FilteredVectorStore con filtro de metadatos
queries = [
    ("technology companies and innovation", {'category': 'Sci/Tech'}),
    ("basketball game and sports competition", {'category': 'Sports'}),
    ("stock market and finance", {'category': 'Business'}),
    ("international politics and war", {'category': 'World'}),
    ("scientific discoveries and artificial intelligence", {'category': 'Sci/Tech'})
]

for query, metadata_filter in queries:

    print("=" * 80)
    print(f"QUERY: {query}")
    print(f"FILTER: {metadata_filter}")
    print("=" * 80)

    results = filtered_vector_store.search(
        query=query,
        top_k=3,
        metadata_filter=metadata_filter
    )

    for i, result in enumerate(results, 1):

        print(f"\nResult #{i}")
        print(f"Score: {result.score:.4f}")

        print("\nMetadata:")
        for key, value in result.document.metadata.items():
            print(f"  {key}: {value}")

        print("\nText:")
        print(result.document.text[:300])

        print("\n" + "-" * 80)

## Reflexiones personales

Esta actividad permitió comprender cómo funcionan internamente los sistemas de búsqueda semántica utilizados en aplicaciones de inteligencia artificial.

Implementar manualmente un VectorStore ayudó a entender el proceso de:
- generación de embeddings
- almacenamiento vectorial
- cálculo de similitud
- recuperación de documentos relevantes

Asimismo, la implementación de FilteredVectorStore mostró la importancia de combinar búsqueda semántica con filtros estructurados basados en metadatos, algo común en sistemas reales de recuperación de información.

Finalmente, la actividad permitió observar cómo los embeddings capturan relaciones semánticas entre textos incluso cuando las palabras exactas no coinciden.
